# Intermediate 09 — Authorization Governance, Delegation & Least Privilege at Scale

## Enterprise scenario

A company operates dozens of agents. Permissions originate from roles, relationships, task grants and agent-to-agent delegation.

We will build a small **authorization governance control plane** that can answer:

```text
What authority exists?
How did an agent get it?
Is it broader than needed?
Does it create a toxic combination?
Is the owner still accountable?
When was it reviewed?
What should be reduced or revoked?
```


In [ ]:
import pandas as pd
import networkx as nx
from datetime import datetime, timedelta, timezone
import json, uuid, copy

NOW=datetime.now(timezone.utc)
def ts(days=0):
    return NOW+timedelta(days=days)


## 1 — Agent inventory

In [ ]:
agents=pd.DataFrame([
 {"agent_id":"agent:claims","owner":"claims-platform","status":"active","risk":"R3","last_seen":ts(-1)},
 {"agent_id":"agent:research","owner":"knowledge-platform","status":"active","risk":"R2","last_seen":ts(-2)},
 {"agent_id":"agent:finance","owner":"finance-platform","status":"active","risk":"R4","last_seen":ts(-1)},
 {"agent_id":"agent:legacy","owner":None,"status":"active","risk":"R3","last_seen":ts(-120)},
])
agents


## 2 — Entitlement catalog

In [ ]:
entitlements=pd.DataFrame([
 {"principal":"agent:claims","action":"claim.read","resource":"claims/*","source":"policy","risk":"medium","last_used":ts(-1)},
 {"principal":"agent:claims","action":"claim.update","resource":"claims/assigned","source":"policy","risk":"high","last_used":ts(-40)},
 {"principal":"agent:claims","action":"claim.delete","resource":"claims/*","source":"legacy-role","risk":"critical","last_used":ts(-200)},
 {"principal":"agent:finance","action":"vendor.create","resource":"vendors/*","source":"role","risk":"high","last_used":ts(-2)},
 {"principal":"agent:finance","action":"payment.create","resource":"payments/*","source":"role","risk":"critical","last_used":ts(-1)},
 {"principal":"agent:finance","action":"payment.approve","resource":"payments/*","source":"role","risk":"critical","last_used":ts(-1)},
])
entitlements


## 3 — Delegation graph

In [ ]:
delegations=[
 {"delegator":"user:alice","delegatee":"agent:claims","actions":{"claim.read","claim.update"},"resource":"claim:483","redelegable":True,"max_depth":1,"expires_at":ts(1)},
 {"delegator":"agent:claims","delegatee":"agent:research","actions":{"document.read"},"resource":"kb:claims","redelegable":False,"max_depth":0,"expires_at":ts(1)},
]
G=nx.DiGraph()
for d in delegations:
    G.add_edge(d["delegator"],d["delegatee"],**d)
print(list(G.edges()))


## 4 — Delegation paths and depth

In [ ]:
for source in G.nodes:
    for target in G.nodes:
        if source!=target and nx.has_path(G,source,target):
            print(source,"->",target,"depth",nx.shortest_path_length(G,source,target))


## 5 — Delegation cycle detection

In [ ]:
cycle_graph=G.copy()
cycle_graph.add_edge("agent:research","user:alice")
print(list(nx.simple_cycles(cycle_graph)))


## 6 — Authority attenuation

In [ ]:
def attenuated(parent_actions,child_actions):
    return set(child_actions) <= set(parent_actions)

print(attenuated({"claim.read","claim.update"},{"claim.read"}))
print(attenuated({"claim.read"},{"claim.read","claim.delete"}))


## 7 — Detect expired delegations

In [ ]:
expired=[d for d in delegations if d["expires_at"] < NOW]
print(expired)


## 8 — Orphaned agents

In [ ]:
orphaned=agents[
    (agents["status"]=="active") &
    (agents["owner"].isna())
]
orphaned


## 9 — Stale agents

In [ ]:
stale_agents=agents[(NOW-agents["last_seen"]).dt.days > 30]
stale_agents


## 10 — Stale entitlements

In [ ]:
entitlements["days_since_use"]=(NOW-entitlements["last_used"]).dt.days
entitlements[entitlements["days_since_use"]>90]


## 11 — Wildcard analysis

In [ ]:
wildcards=entitlements[
 entitlements["resource"].str.contains(r"\*",regex=True)
]
wildcards


## 12 — Risk-weighted wildcard findings

In [ ]:
HIGH={"high","critical"}
wildcards[wildcards["risk"].isin(HIGH)]


## 13 — Granted vs observed

In [ ]:
usage=pd.DataFrame([
 {"principal":"agent:claims","action":"claim.read","uses_90d":8240},
 {"principal":"agent:claims","action":"claim.update","uses_90d":8},
 {"principal":"agent:claims","action":"claim.delete","uses_90d":0},
 {"principal":"agent:finance","action":"vendor.create","uses_90d":21},
 {"principal":"agent:finance","action":"payment.create","uses_90d":410},
 {"principal":"agent:finance","action":"payment.approve","uses_90d":409},
])
review=entitlements.merge(usage,on=["principal","action"],how="left").fillna({"uses_90d":0})
review[["principal","action","resource","risk","uses_90d"]]


## 14 — Least-privilege candidates

In [ ]:
candidates=review[
 (review["uses_90d"]==0) |
 ((review["days_since_use"]>90) & review["risk"].isin(HIGH))
]
candidates


These are **review candidates**, not automatic revocations. Rare but legitimate emergency permissions may have zero observed use.

## 15 — Toxic combinations

In [ ]:
TOXIC=[
 {"name":"self-approved-payment","permissions":{"payment.create","payment.approve"}},
 {"name":"vendor-and-payment-control","permissions":{"vendor.create","payment.create","payment.approve"}},
]

def toxic_for(principal):
    held=set(entitlements[entitlements.principal==principal].action)
    return [t["name"] for t in TOXIC if t["permissions"] <= held]

for p in entitlements.principal.unique():
    print(p,toxic_for(p))


## 16 — Separation of duties

In [ ]:
assert "self-approved-payment" in toxic_for("agent:finance")
print("SoD violation detected for finance agent")


## 17 — Transitive authority graph

In [ ]:
authority=nx.DiGraph()
authority.add_edges_from([
 ("agent:orchestrator","agent:vendor"),
 ("agent:orchestrator","agent:payment"),
 ("agent:vendor","perm:vendor.create"),
 ("agent:payment","perm:payment.approve"),
])
for perm in ["perm:vendor.create","perm:payment.approve"]:
    print(perm,nx.has_path(authority,"agent:orchestrator",perm))


## 18 — Toxic transitive paths

In [ ]:
def reachable_permissions(graph,principal):
    return {n.replace("perm:","") for n in nx.descendants(graph,principal) if n.startswith("perm:")}

rp=reachable_permissions(authority,"agent:orchestrator")
print(rp)
print("toxic:", {"vendor.create","payment.approve"} <= rp)


## 19 — Temporary exceptions

In [ ]:
exceptions=pd.DataFrame([
 {"id":"exc:1","principal":"agent:claims","permission":"claims.bulk_export","approved_by":"security","expires_at":ts(2),"controls":["human approval"]},
 {"id":"exc:old","principal":"agent:legacy","permission":"claims.export","approved_by":"security","expires_at":ts(-30),"controls":["audit review"]},
])
exceptions


## 20 — Expired exceptions

In [ ]:
exceptions[exceptions["expires_at"]<NOW]


## 21 — Risk-based review cadence

In [ ]:
CADENCE={"R1":365,"R2":180,"R3":90,"R4":30}
reviews=pd.DataFrame([
 {"agent_id":"agent:claims","last_review":ts(-100)},
 {"agent_id":"agent:research","last_review":ts(-100)},
 {"agent_id":"agent:finance","last_review":ts(-35)},
 {"agent_id":"agent:legacy","last_review":ts(-200)},
])
r=agents.merge(reviews,on="agent_id")
r["review_age"]=(NOW-r["last_review"]).dt.days
r["cadence"]=r["risk"].map(CADENCE)
r["overdue"]=r["review_age"]>r["cadence"]
r[["agent_id","risk","review_age","cadence","overdue"]]


## 22 — Recertification packet

In [ ]:
def review_packet(agent_id):
    a=agents[agents.agent_id==agent_id].iloc[0].to_dict()
    e=review[review.principal==agent_id].to_dict("records")
    return {
      "agent":a,
      "entitlements":e,
      "recommendations":[
        {"action":"review_for_removal","permission":x["action"]}
        for x in e if x.get("uses_90d",0)==0
      ]
    }

print(json.dumps(review_packet("agent:claims"),indent=2,default=str))


## 23 — Recertification outcomes

In [ ]:
ALLOWED_OUTCOMES={"renew","reduce","revoke","expire","escalate"}

decision={
 "review_id":"review:100",
 "agent":"agent:claims",
 "permission":"claim.delete",
 "outcome":"revoke",
 "reviewer":"claims-owner",
 "reason":"unused and outside current operating model",
 "timestamp":NOW
}
assert decision["outcome"] in ALLOWED_OUTCOMES
decision


## 24 — Policy versions

In [ ]:
policy_versions=[
 {"version":"v18","approved":True,"deployed_at":ts(-30),"permissions":{"claim.read","claim.update"}},
 {"version":"v19","approved":False,"deployed_at":None,"permissions":{"claim.read","claim.update","claim.export"}},
]


## 25 — Policy impact analysis

In [ ]:
old=policy_versions[0]["permissions"]
new=policy_versions[1]["permissions"]
print("newly allowed:",new-old)
print("removed:",old-new)


## 26 — Expansion gate

In [ ]:
HIGH_RISK_ACTIONS={"claim.export","claim.delete","payment.create","payment.approve"}
expansion=(new-old) & HIGH_RISK_ACTIONS
print("requires explicit security review:",bool(expansion),expansion)


## 27 — Governance findings

In [ ]:
findings=[]

for _,a in orphaned.iterrows():
    findings.append({"severity":"high","code":"ORPHANED_AGENT","subject":a.agent_id})

for _,e in candidates.iterrows():
    findings.append({"severity":e.risk,"code":"LEAST_PRIVILEGE_REVIEW","subject":e.principal,"permission":e.action})

for p in entitlements.principal.unique():
    for combo in toxic_for(p):
        findings.append({"severity":"critical","code":"TOXIC_COMBINATION","subject":p,"combination":combo})

for _,x in exceptions[exceptions.expires_at<NOW].iterrows():
    findings.append({"severity":"high","code":"EXPIRED_EXCEPTION","subject":x.principal,"exception":x.id})

pd.DataFrame(findings)


## 28 — Governance metrics

In [ ]:
metrics={
 "active_agents":int((agents.status=="active").sum()),
 "orphaned_agents":len(orphaned),
 "stale_agents":len(stale_agents),
 "entitlements":len(entitlements),
 "least_privilege_candidates":len(candidates),
 "toxic_combinations":sum(len(toxic_for(p)) for p in entitlements.principal.unique()),
 "expired_exceptions":int((exceptions.expires_at<NOW).sum()),
 "high_risk_wildcards":len(wildcards[wildcards.risk.isin(HIGH)]),
}
metrics


## 29 — Permission-creep baseline

In [ ]:
baseline={
 "agent:claims":{"claim.read","claim.update"}
}
current=set(entitlements[entitlements.principal=="agent:claims"].action)
print("creep:",current-baseline["agent:claims"])


## 30 — Offboarding plan

In [ ]:
SURFACES=[
 "oauth_grants","api_keys","workload_identity","spiffe_registration",
 "tool_access","mcp_registration","delegations","policy_bindings",
 "secrets","scheduled_jobs"
]

def offboard(agent_id):
    return {
      "agent":agent_id,
      "status":"retiring",
      "required_revocations":SURFACES,
      "verification":"confirm zero effective authority after cleanup"
    }

print(json.dumps(offboard("agent:legacy"),indent=2))


## 31 — Shadow-agent discovery

In [ ]:
registered=set(agents.agent_id)
observed={
 "agent:claims",
 "agent:research",
 "agent:finance",
 "agent:unknown-automation"
}
print("shadow agents:",observed-registered)


## 32 — Historical decision evidence

In [ ]:
evidence={
 "decision_id":str(uuid.uuid4()),
 "principal":"agent:claims",
 "action":"claim.update",
 "resource":"claim:483",
 "policy_version":"v18",
 "authorization_model_version":"fga:7",
 "entitlement_snapshot":"snapshot:2026-08-18T23:00Z",
 "delegation_path":["user:alice","agent:claims"],
 "decision":"allow"
}
print(json.dumps(evidence,indent=2))


## 33 — Adversarial tests

In [ ]:
# Orphaned active agent must be found.
assert "agent:legacy" in set(orphaned.agent_id)

# Critical unused legacy permission must be reviewed.
assert ((candidates.principal=="agent:claims") & (candidates.action=="claim.delete")).any()

# Agent cannot both create and approve payments unnoticed.
assert "self-approved-payment" in toxic_for("agent:finance")

# Expired exception must be detected.
assert "exc:old" in set(exceptions[exceptions.expires_at<NOW].id)

# Shadow identity must be visible.
assert "agent:unknown-automation" in observed-registered

print("all governance regression tests passed")


## 34 — OpenFGA lab

Use the current OpenFGA documentation to model:

```text
user
agent
task
tool
claim
organization
```

Relationships:

```text
user:alice operator agent:claims
agent:claims assigned task:483
task:483 resource claim:483
agent:claims allowed_tool tool:claim-reader
```

Then test:

1. direct permission;
2. inherited organization permission;
3. user→agent acting-for relationship;
4. delegation to a sub-agent;
5. revoked relationship;
6. cross-tenant negative test;
7. maximum delegation-depth enforcement in surrounding policy.

OpenFGA is a relationship engine; do not force every dynamic risk signal into durable tuples.


## 35 — Cedar lab

Use:

```text
policies/cedar/governance.cedar
```

Build a typed schema for:

```text
User
Agent
Task
Claim
Payment
Tool
```

Validate policies against the schema.

Test Cedar semantics including:

```text
default deny
permit
forbid-overrides-permit
missing/malformed context
```

Use Cedar context for transient facts such as assurance/risk rather than turning every request fact into a persistent entity.


## 36 — OPA governance lab

Use:

```text
policies/opa/governance.rego
```

Feed the governance inventory to OPA and add rules for:

```text
delegation depth
wildcards
orphaned agents
expired exceptions
overdue reviews
toxic combinations
unapproved high-risk expansion
```

Return structured findings with reason codes and remediation guidance.


## 37 — Production extension

Replace the notebook DataFrames with feeds from:

```text
IdP / IAM
OAuth authorization server
SPIFFE/SPIRE
OpenFGA or other FGA system
Cedar/OPA policy stores
MCP registry/gateway
API gateway
cloud IAM
agent registry
decision logs
SIEM
```

Normalize them into one governance graph.

The difficult enterprise problem is rarely *writing one policy*. It is understanding all paths by which authority exists.


## 38 — Review questions

1. What is authorization governance?
2. Why is runtime authorization alone insufficient?
3. What belongs in an agent entitlement catalog?
4. What is effective authority?
5. Why model delegation as a graph?
6. What metadata belongs on delegation edges?
7. What is authority attenuation?
8. What is permission drift?
9. What is an orphaned agent?
10. What is a toxic permission combination?
11. How can SoD fail through multiple collaborating agents?
12. What is ReBAC?
13. When is OpenFGA useful?
14. When is a policy engine more appropriate than a relationship engine?
15. What are Cedar's principal/action/resource/context concepts?
16. Why are Cedar schemas useful?
17. How can OPA support governance?
18. Why should policy be versioned?
19. What is policy impact analysis?
20. What should an access-review packet contain?
21. Why is zero usage not sufficient proof for revocation?
22. Why should exceptions expire?
23. What should agent offboarding revoke?
24. How can shadow agents be discovered?
25. What is a PDP?
26. What is a PEP?
27. Why are bypass tests necessary?
28. What should happen when the authorization service is unavailable?
29. Why can decision caching undermine revocation?
30. What evidence is needed to reproduce a historical authorization decision?

# Next course

## Intermediate 10 — Authorization Observability & Audit Analytics for Agents
